In [1]:
# === IMPORTS ===
import os
import json
import pandas as pd
import omama as O
from omama import DataHelper, DeepSight

# === CONFIGURATION ===
base_dir = "/raid/data01/deephealth/dh_dh0new/"
output_csv = "/raid/mpsych/OMAMA/2025/PLAYGROUND/deepsight_4vs1_image_analysis_25patients.csv"

studies = os.listdir(base_dir)
studies = sorted(studies)[:25]  # <--- 🚀 Only first 25 studies now
all_predictions = []

# === MAIN LOOP ===
for study in studies:
    study_path = os.path.join(base_dir, study)
    if not os.path.isdir(study_path):
        continue
    
    # Get only DICOMs (DXm prefix)
    dicoms = [f for f in os.listdir(study_path) if f.startswith('DXm')]
    dicom_paths = [os.path.join(study_path, d) for d in dicoms]
    
    if len(dicom_paths) < 4:
        print(f"Skipping {study} (not enough images)")
        continue
    
    # Load images
    images = []
    for path in dicom_paths:
        img = DataHelper.get(image=os.path.basename(path))
        images.append(img)
    
    # --- Run DeepSight on 4 images together ---
    preds_4img = DeepSight.run(images)
    for sop_uid, pred in preds_4img.items():
        all_predictions.append({
            "study_uid": study,
            "sop_uid": sop_uid,
            "mode": "4-image",
            "score": pred['score'],
            "num_rois": len(pred['coords']) if pred['coords'] else 0,
            "passed_acceptance_criteria": pred['score'] != -1
        })
    
    # --- Run DeepSight separately on each image ---
    for img in images:
        preds_single = DeepSight.run([img])
        sop_uid = img.SOPInstanceUID
        pred = preds_single.get(sop_uid, None)
        if pred:
            all_predictions.append({
                "study_uid": study,
                "sop_uid": sop_uid,
                "mode": "1-image",
                "score": pred['score'],
                "num_rois": len(pred['coords']) if pred['coords'] else 0,
                "passed_acceptance_criteria": pred['score'] != -1
            })

# === SAVE TO CSV ===
df = pd.DataFrame(all_predictions)
df.to_csv(output_csv, index=False)

print(f"✅ Done! Results saved at: {output_csv}")


Loading config data from ini file
DataLoader type is:  <class 'omama.loaders.omama_loader.OmamaLoader'>
Running DeepSight on 4 cases, please be patient...
Running DeepSight on 4 cases, please be patient...
Running DeepSight on 4 cases, please be patient...
Running DeepSight on 4 cases, please be patient...
Running DeepSight on 4 cases, please be patient...
Running DeepSight on 5 cases, please be patient...
Running DeepSight on 5 cases, please be patient...
Running DeepSight on 4 cases, please be patient...
Running DeepSight on 4 cases, please be patient...
Running DeepSight on 6 cases, please be patient...
Running DeepSight on 4 cases, please be patient...
Running DeepSight on 4 cases, please be patient...
Running DeepSight on 4 cases, please be patient...
Running DeepSight on 4 cases, please be patient...
Running DeepSight on 4 cases, please be patient...
Running DeepSight on 4 cases, please be patient...
Running DeepSight on 5 cases, please be patient...
Running DeepSight on 4 cases,

In [5]:
# === IMPORTS ===
import pandas as pd
import matplotlib.pyplot as plt  # Still needed if you ever plot manually
from scipy.stats import ttest_rel

# === CONFIGURATION ===
csv_file = "/raid/mpsych/OMAMA/2025/PLAYGROUND/deepsight_4vs1_image_analysis_25patients.csv"

# === STEP 1: Load CSV ===
df = pd.read_csv(csv_file)
print(f"Loaded {len(df)} predictions ✅")

# === STEP 2: Separate Data ===
df_4img = df[df['mode'] == '4-image']
df_1img = df[df['mode'] == '1-image']

# === STEP 3: Summary Statistics ===
print("\nSummary for 4-image mode:")
print(df_4img['score'].describe())

print("\nSummary for 1-image mode:")
print(df_1img['score'].describe())

# === STEP 4: Paired Analysis ===
# Match scores by SOPInstanceUID for fair comparison
merged = df_4img.merge(df_1img, on="sop_uid", suffixes=("_4img", "_1img"))

print(f"\nFound {len(merged)} matching SOPs between 4-image and 1-image mode.")

# === STEP 5: Paired T-Test ===
t_stat, p_val = ttest_rel(merged['score_4img'], merged['score_1img'])
print(f"\nPaired t-test results:")
print(f"  t-statistic = {t_stat:.4f}")
print(f"  p-value = {p_val:.4f}")

if p_val < 0.05:
    print("✅ Statistically significant difference between 4 images and 1 image predictions!")
else:
    print("⚡ No statistically significant difference found (p > 0.05).")

# === DONE ===
print("\n✅ Analysis complete!")


Loaded 214 predictions ✅

Summary for 4-image mode:
count    107.000000
mean       0.203284
std        0.173360
min        0.033379
25%        0.083524
50%        0.130306
75%        0.252881
max        0.725285
Name: score, dtype: float64

Summary for 1-image mode:
count    107.000000
mean       0.203284
std        0.173360
min        0.033379
25%        0.083524
50%        0.130306
75%        0.252881
max        0.725285
Name: score, dtype: float64

Found 107 matching SOPs between 4-image and 1-image mode.

Paired t-test results:
  t-statistic = nan
  p-value = nan
⚡ No statistically significant difference found (p > 0.05).

✅ Analysis complete!


In [7]:
print("")